In [3]:
import sys
import os
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg
import scipy.interpolate as spi
from scipy.integrate import quad

# Get the current working directory (Week_2) and step up one level to the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now Python can find the 'src' folder
from src.pricing.models import (
    heston_mc_european_call, 
    heston_semi_analytic, 
    heston_tree_european_call, 
    heston_fdm_european_call
)

In [6]:
import time
import numpy as np
import pandas as pd

def run_benchmarks():
    # Standard Heston test parameters
    S0 = 100.0
    T = 1.0
    r = 0.05
    kappa = 2.0
    theta = 0.04
    omega = 0.2
    rho = -0.5
    V0 = 0.04
    
    # Testing across In-The-Money, At-The-Money, and Out-Of-The-Money strikes
    strikes = [90.0, 100.0, 110.0]
    
    # 将 Semi-Analytic 放在第一位，它就会出现在表格的第一行
    results = {
        "Semi-Analytic (Exact)": {"abs_errors": [], "rel_errors": [], "times": []},
        "Simulation (baseline)": {"abs_errors": [], "rel_errors": [], "times": []},
        "Binomial Tree": {"abs_errors": [], "rel_errors": [], "times": []},
        "Finite Difference": {"abs_errors": [], "rel_errors": [], "times": []}
    }
    
    print("Running benchmarks across strikes:", strikes)
    
    for K in strikes:
        # 1. Semi-Analytic (Ground Truth / Exact Price)
        start = time.time()
        true_price = heston_semi_analytic(S0, K, T, r, V0, kappa, theta, omega, rho)
        sa_time = time.time() - start
        
        results["Semi-Analytic (Exact)"]["abs_errors"].append(0.0)
        results["Semi-Analytic (Exact)"]["rel_errors"].append(0.0)
        results["Semi-Analytic (Exact)"]["times"].append(sa_time)

        # 2. Monte Carlo Simulation (Baseline)
        start = time.time()
        mc_price, _ = heston_mc_european_call(S0, K, T, r, kappa, theta, omega, rho, V0, n_steps=100, n_sims=50000)
        mc_time = time.time() - start
        
        results["Simulation (baseline)"]["abs_errors"].append(abs(mc_price - true_price))
        results["Simulation (baseline)"]["rel_errors"].append(abs(mc_price - true_price) / true_price)
        results["Simulation (baseline)"]["times"].append(mc_time)
        
        # 3. Binomial Tree 
        start = time.time()
        tree_price = heston_tree_european_call(S0, K, T, r, kappa, theta, omega, rho, V0, n=10, mv=70, mz=70)
        tree_time = time.time() - start
        
        results["Binomial Tree"]["abs_errors"].append(abs(tree_price - true_price))
        results["Binomial Tree"]["rel_errors"].append(abs(tree_price - true_price) / true_price)
        results["Binomial Tree"]["times"].append(tree_time)
        
        # 4. Finite Difference Method
        start = time.time()
        fdm_price = heston_fdm_european_call(S0, K, T, r, kappa, theta, omega, rho, V0, 
                                             N_S=100, N_V=100, N_T=100, S_max=200.0, V_max=1.0)
        fdm_time = time.time() - start
        
        results["Finite Difference"]["abs_errors"].append(abs(fdm_price - true_price))
        results["Finite Difference"]["rel_errors"].append(abs(fdm_price - true_price) / true_price)
        results["Finite Difference"]["times"].append(fdm_time)
        
    # Compile the final table
    table_data = []
    for method, data in results.items():
        abs_errs = np.array(data["abs_errors"])
        rel_errs = np.array(data["rel_errors"]) * 100 
        times = np.array(data["times"])
        
        table_data.append({
            "Method": method,
            "Average Absolute Error": np.mean(abs_errs),
            "Absolute Error Variance": np.var(abs_errs, ddof=1), 
            "Average Relative Error": f"{np.mean(rel_errs):.6f}%",
            "Relative Error Variance": f"{np.var(rel_errs, ddof=1):.6f}%",
            "Average Run Time (s)": f"{np.mean(times):.4f}"
        })
        
    df = pd.DataFrame(table_data)
    
    # Format absolute errors to scientific notation
    df["Average Absolute Error"] = df["Average Absolute Error"].apply(lambda x: f"{x:.6e}")
    df["Absolute Error Variance"] = df["Absolute Error Variance"].apply(lambda x: f"{x:.6e}")
    
    return df

# Run and display the table
benchmark_table = run_benchmarks()
display(benchmark_table)

Running benchmarks across strikes: [90.0, 100.0, 110.0]


,Method,Average Absolute Error,Absolute Error Variance,Average Relative Error,Relative Error Variance,Average Run Time (s)
0,Semi-Analytic (Exact),0.000000e+00,0.000000e+00,0.000000%,0.000000%,0.0038
1,Simulation (baseline),2.806930e-02,2.892544e-04,0.341216%,0.067172%,0.2348
2,Binomial Tree,2.049294e-01,1.246198e-03,2.322256%,1.936355%,0.0083
3,Finite Difference,2.518295e-02,4.510536e-05,0.295200%,0.040350%,6.3361
